# Chapter 4. Discretization of Continuous-Time Dynamic Models 

General SDE format:
$$  \frac{d\mathbf{x_t}}{dt} = \mathbf{a}(\mathbf{x}_t) + \mathbf{L} \mathbf{\omega}(t)$$

We'll write $\mathbf{x_t}$ as $x_t$ in below text.

where $\mathbf{a}(x)$ is a function, $\mathbf{L}$ is a known matrix and $\mathbf{\omega}_t$ is a white noise process with spectral density $Q^c$. 

- In case $a(x)$ is linear, it can be written as $a(x) = F x$, there is a (nearly) closed form for the transition density in discrete time:
$$ p \left( x(t_k)| x(t_{k-1}) \right) = N\left( x(t_{k}) | exp(\mathbf{F} \Delta t_{k-1}) x(t_{k-1}), Q_{k-1}\right) $$
where $\Delta t_{k-1} = t_k - t_{k-1}$, and
$$ Q_{k-1} = \int_0^{\Delta t_{k-1}} exp(\mathbf{F} s) \mathbf{L} \mathbf{Q}^c \mathbf{L}^T exp(\mathbf{F} s)^T ds $$

Note that $exp(.)$ is the matrix exponential function, not an element-wise exponential.

Consequently, discretization gives a linear and Gaussian model:
$$ x_k = \mathbf{A}_{k-1} x_{k-1} + \mathbf{q}_{k-1}$$
where $\mathbf{A}_{k-1} = exp(\mathbf{F} \Delta t_{k-1})$ and $\mathbf{q}_{k-1} \sim N(0, \mathbf{Q}_{k-1}) $.


### Proof for linear case: use Lemme A.9 in Appendix A.4 by assuming $b(x)=0$

Lemma A.9 (Analytical solution to affine SDE). Suppose $x(t)$ obeys the stochastic differential equation (SDE)
$$ \frac{dx(t)}{dt} = F x(t) + b(t) + Lw(t)$$
where $F$ is constant, $b(t)$ is a deterministic function of time, and $w(t)$ is white Gaussian noise with below moments: 
$$ E(w(t)) = 0 $$
$$ Cov(w(\tau_1), w(\tau_2)) = \delta(\tau_1 - \tau_2) Q^c $$
where $\delta(.)$ is a Diract function.

The solution of this SDE then holds:

$$ x(t+\tau) = exp(F\tau) x(t) + \int_t^{t+\tau} exp\left( F(t+\tau - v) \right) b(v) dv + \int_t^{t+\tau} exp\left( F (t+\tau - v) \right) L w(v)dv$$

where $\int_t^{t+\tau} exp\left( F (t+\tau - v) \right) L w(v)dv$ is a Gaussian randmo variable with zero mean and covariance $\int_0^\tau exp(Fv) L Q^c L^T exp(Fv)^T dv$. 

Consequently, $x(t+\tau)| x(t)$ is Gaussian with the moments 


- In case $a(x)$ is non-linear, we can use Euler-Maruyama Method for discretization, where we approximate $\mathbf{a}(x(t_k))$ as $\mathbf{a}(x(t_{k-1}))$, so we obtain:
$$  \frac{d\mathbf{x}(t)}{dt} \approx \mathbf{a}(\mathbf{x}(t_{k-1})) + \mathbf{L} \mathbf{\omega}(t)$$

Hence:
$$x_t = \mathbf{F}_{k-1}(x_{k-1}) + q_{k-1}$$

where:
$$\mathbf{F}_{k-1}(x_{k-1}) = x_{k-1} +  \mathbf{a}(\mathbf{x}_{k-1}) \Delta t_{k-1}$$
$$q_{k-1} \sim N(0, Q_{k-1})$$
$$Q_{k-1} = \mathbf{L} Q^C \mathbf{L}^T \Delta t_{k-1} $$

It's more convenient to rewrite it to below format:
$$ x_k = \mathbf{A}_{k-1} x_{k-1} + \mathbf{L} \widetilde{\mathbf{q}}_{k-1}$$

where:
$$ \widetilde{\mathbf{q}}_{k-1} \sim N(0, \widetilde{Q}_{k-1}) $$
$$ \widetilde{Q}_{k-1} = Q^C \Delta t_{k-1} $$



### Algorithm 4.12 (Simulating SDE solution with Euler-Maruyama)
Split interval $[t_{k-1}, t_k]$ into n steps of length $\Delta \tau = (t_k - t_{k-1})/n$, then do the following:
- Start simulation from the state $\hat{x}_0 = x(t_{k-1})$
- For $j=1, ..., n$, simulate simulation using Euler-Maruyama:
  $$ \hat{x}_j = x_{j-1} +  \mathbf{a}(\hat{\mathbf{x}}_{j-1}) \Delta \tau + \mathbf{L} \widetilde{\mathbf{q}}_{j-1} $$
  where $\widetilde{\mathbf{q}}_{j-1} $ is a random walk from $N(0, Q^C \Delta \tau )$
- The approximation of the trajectory $x(t_{k-1}) \rightarrow x(t_k) $ is then given by the piece-wise constant trajectory consisting of the points $\{ \hat{x}_0, \hat{x}_1, ..., \hat{x}_n \} $.

## 4.1 

Use Algorithm 4.2 to sample from a scalar Wiener process:

$$ \frac{dx(t)}{dt} = \omega_t$$

where $\omega_t$ has the spectral density  $q^c = 1$. Use $\Delta \tau =0.01$ to generate and illustrate $N=3$ samples in the interval $t \in [0,4]$ when $x(0) = 0$.

In [38]:
import numpy as np
import pandas as pd
import matplotlib as plt


x_0 = 0
N = 3
T = 4 
tau = 0.01

X = np.zeros((T+1, 1))
X_hat = np.zeros((T+1, N+1))

for i in range(1, T+1):
    X_hat[i, 0] = X[i-1, 0]
    for j in range(1, N+1):
        omega = np.random.normal(loc = 0, scale = tau)
        X_hat[i, j] = X_hat[i, j-1] + omega 

        if j == N:
            print(j)
            X[i, 0] = X_hat[i,j]
        


3
3
3
3


In [39]:
X_hat

array([[ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , -0.00570171, -0.00407435,  0.00104483],
       [ 0.00104483,  0.00368265,  0.00445809,  0.0142933 ],
       [ 0.0142933 ,  0.01465047,  0.0026139 ,  0.00295853],
       [ 0.00295853,  0.00090562, -0.00015655,  0.000333  ]])

In [40]:
X

array([[0.        ],
       [0.00104483],
       [0.0142933 ],
       [0.00295853],
       [0.000333  ]])

In [41]:
X_hat.flatten()

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
       -0.00570171, -0.00407435,  0.00104483,  0.00104483,  0.00368265,
        0.00445809,  0.0142933 ,  0.0142933 ,  0.01465047,  0.0026139 ,
        0.00295853,  0.00295853,  0.00090562, -0.00015655,  0.000333  ])

In [9]:
X_hat

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])